# Steerling-8B Instruct: Logit Contribution Analysis

Logit decomposition captured **during generation** — each token's
contribution comes from the forward pass where it was committed, using the
actual partially-masked context.

For each predicted token $y$, we decompose its logit into three components:

$$\text{logit}(y) = \underbrace{\text{known} \cdot W_y}_{\text{known concepts}} + \underbrace{\hat{u} \cdot W_y}_{\text{discovered concepts}} + \underbrace{\varepsilon \cdot W_y}_{\text{residual}}$$

- **Known**: contribution from the top-k learned concept embeddings (human-interpretable features)
- **Discovered**: contribution from the factorized residual head (captures what known concepts miss)
- **Epsilon**: small correction term that preserves reconstruction fidelity

**Requirements:** Interpretable Steerling model, GPU with >= 18 GB VRAM

## 1. Load Model

In [ ]:
import torch
import numpy as np
from transformers import AutoModel, AutoTokenizer
from steerling import SteerlingGenerator, GenerationConfig
from steerling.inference.causal_diffusion import GenerationStepInfo

model_id = "guidelabs/steerling-8b-instruct"

model = AutoModel.from_pretrained(model_id, trust_remote_code=True, dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
generator = SteerlingGenerator.from_model(model, tokenizer, device="cuda")

print(generator)
print(f"Interpretable: {generator.is_interpretable}")

## 2. Decomposition

For each committed token, we capture `known_predicted`, `unk_hat`, and `unk_for_lm`
from the forward pass at commit time. We then compute each component's dot product
with the LM head row $W_y$ for the committed token $y$.

This uses the actual partially-masked context the model operated in,
not a post-hoc forward pass over the fully-unmasked sequence.

In [ ]:
@torch.no_grad()
def logit_contribution(generator, prompt, max_new_tokens=128, steps=128, temperature=0.0, stop_tokens=None):
    """
    Generate text and decompose each committed token's logit into
    known, discovered (unk_hat), and epsilon contributions.
    """
    config = GenerationConfig(
        max_new_tokens=max_new_tokens, steps=steps, temperature=temperature,
        stop_tokens=stop_tokens or [],
    )

    lm_weight = generator.model.transformer.lm_head.weight  # (V, D)

    # Buffers to accumulate per-token contributions
    all_positions = []
    all_token_ids = []
    all_known_c = []
    all_discovered_c = []
    all_eps_c = []

    def on_commit(info: GenerationStepInfo):
        outputs = info.outputs
        pos = info.committed_positions
        tids = info.committed_token_ids

        known = outputs.known_predicted[0, pos]      # (P, D)
        discovered = outputs.unk_hat[0, pos] if outputs.unk_hat is not None else torch.zeros_like(known)
        unk_for_lm = outputs.unk_for_lm[0, pos]     # (P, D)
        epsilon = unk_for_lm - discovered            # (P, D)

        W_y = lm_weight[tids].float()                # (P, D)

        all_positions.append(pos.cpu())
        all_token_ids.append(tids.cpu())
        all_known_c.append((known.float() * W_y).sum(dim=-1).cpu())
        all_discovered_c.append((discovered.float() * W_y).sum(dim=-1).cpu())
        all_eps_c.append((epsilon.float() * W_y).sum(dim=-1).cpu())

    gen_output = generator.generate_full(prompt, config, step_callback=on_commit)
    prompt_len = gen_output.prompt_tokens

    # Concatenate and sort by position
    positions = torch.cat(all_positions)
    token_ids = torch.cat(all_token_ids)
    known_c = torch.cat(all_known_c)
    discovered_c = torch.cat(all_discovered_c)
    eps_c = torch.cat(all_eps_c)

    order = positions.argsort()
    positions = positions[order]
    token_ids = token_ids[order]
    known_c = known_c[order]
    discovered_c = discovered_c[order]
    eps_c = eps_c[order]

    # Filter to generated tokens only (after prompt, skip mask/eos)
    mask_id = generator.mask_token_id
    eos_id = generator.eos_token_id
    valid = (positions >= prompt_len) & (token_ids != mask_id) & (token_ids != eos_id)

    positions = positions[valid]
    token_ids = token_ids[valid]
    known_c = known_c[valid]
    discovered_c = discovered_c[valid]
    eps_c = eps_c[valid]

    # Absolute fractions
    abs_sum = (known_c.abs() + discovered_c.abs() + eps_c.abs()).clamp(min=1e-8)

    return {
        "text": gen_output.text,
        "prompt": prompt,
        "token_ids": token_ids,
        "tokens": [tokenizer.decode([t]) for t in token_ids.tolist()],
        "known_frac": (known_c.abs() / abs_sum).numpy(),
        "discovered_frac": (discovered_c.abs() / abs_sum).numpy(),
        "eps_frac": (eps_c.abs() / abs_sum).numpy(),
        "known_c": known_c.numpy(),
        "discovered_c": discovered_c.numpy(),
        "eps_c": eps_c.numpy(),
    }

## 3. Per-Token Breakdown

Apply the chat template and generate. The table shows the fractional contribution
of each component for every generated token.

In [ ]:
def show_contribution(result, max_tokens=40):
    """Print a compact summary of logit contributions."""
    print(f"Prompt:    {result['prompt'][:200]}")
    print(f"Generated: {result['text'][:200]}")
    print()

    n = min(len(result['tokens']), max_tokens)
    kf = result['known_frac'][:n]
    df = result['discovered_frac'][:n]
    ef = result['eps_frac'][:n]

    print(f"{'Token':<15} {'Known':>7} {'Disc.':>7} {'Eps':>7}")
    print("-" * 40)
    for i in range(n):
        tok = repr(result['tokens'][i])
        print(f"{tok:<15} {kf[i]:6.1%} {df[i]:6.1%} {ef[i]:6.1%}")

    print("-" * 40)
    print(f"{'Mean':<15} {kf.mean():6.1%} {df.mean():6.1%} {ef.mean():6.1%}")

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful AI assistant. Be concise."},
    {"role": "user", "content": "What is a diffusion language model?"},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")

torch.manual_seed(42)
result = logit_contribution(
    generator, prompt,
    max_new_tokens=256, steps=256,
    stop_tokens=[eot_id],
)
show_contribution(result)

## 4. Overall Contribution

A stacked bar chart showing the mean contribution of each component across all generated tokens.
This gives a quick summary of how much the model relies on known concepts vs discovered concepts
for a given prompt.

In [ ]:
import matplotlib.pyplot as plt

PURPLE = "#675BF2"
PURPLE_LIGHT = "#ceb4fe"
GOLD = "#e2bc26"

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: per-token stacked bar ---
n = min(len(result["tokens"]), 40)
tokens = [result["tokens"][i][:8] for i in range(n)]
x = np.arange(n)

axes[0].bar(x, result["known_frac"][:n], label="Known", color=PURPLE)
axes[0].bar(x, result["discovered_frac"][:n], bottom=result["known_frac"][:n], label="Discovered", color=PURPLE_LIGHT)
axes[0].bar(
    x,
    result["eps_frac"][:n],
    bottom=result["known_frac"][:n] + result["discovered_frac"][:n],
    label="Epsilon",
    color=GOLD,
)
axes[0].set_xticks(x)
axes[0].set_xticklabels(tokens, rotation=90, fontsize=7)
axes[0].set_ylabel("Fraction of |logit|")
axes[0].set_title("Per-Token Logit Contribution")
axes[0].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
axes[0].set_ylim(0, 1.05)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

# --- Right: overall mean bar ---
means = [result["known_frac"].mean(), result["discovered_frac"].mean(), result["eps_frac"].mean()]
labels = ["Known", "Discovered", "Epsilon"]
colors = [PURPLE, PURPLE_LIGHT, GOLD]

bars = axes[1].bar(labels, means, color=colors, width=0.5)
for bar, val in zip(bars, means):
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01, f"{val:.1%}",
                 ha="center", fontsize=11)
axes[1].set_ylabel("Mean Fraction of |logit|")
axes[1].set_title("Average Contribution Across All Tokens")
axes[1].set_ylim(0, max(means) * 1.25)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.suptitle(f'Logit Decomposition (Instruct)', fontsize=11)
plt.tight_layout()
plt.show()